# 08 — RAG Generation with AMALIA

This notebook implements the complete end-to-end **CLARA** Retrieval-Augmented Generation (RAG) workflow.

The previous notebooks prepared and evaluated the retrieval component. This final experimental stage combines semantic retrieval from **Qdrant** with answer generation using **AMALIA-9B**, served locally through **Ollama**.

The objective is to answer natural-language questions using the retrieved Portuguese legal provisions as grounding context.

The complete workflow is:

```text
User Question
      │
      ▼
   BGE-M3
      │
      ▼
Question Embedding
      │
      ▼
    Qdrant
      │
      ▼
Top-K Legal Chunks
      │
      ▼
Context Construction
      │
      ▼
  AMALIA-9B
      │
      ▼
Grounded Answer
```

## 1. Imports

This block loads the libraries required for HTTP communication with Ollama and Qdrant, JSON processing, timing, and inspection of the generated RAG results.

In [1]:
import json
import time

import pandas as pd
import requests


## 2. RAG Configuration

The main runtime parameters of the end-to-end RAG pipeline are defined here.

These include the Ollama endpoint, Qdrant endpoint, embedding model, AMALIA model, vector collection, Top-K retrieval value, generation temperature, and request timeouts.

The configuration should remain aligned with the settings used by the production CLARA API.

In [2]:
# ---------------------------------------------------------
# Configuração
# ---------------------------------------------------------

OLLAMA_URL = "http://localhost:11434"
QDRANT_URL = "http://localhost:6333"

EMBEDDING_MODEL = "bge-m3"
COLLECTION_NAME = "clara_bge_m3"

TOP_K = 5

# Se souberes o tag exacto do AMALIA no Ollama,
# coloca-o aqui. Exemplo:
# AMALIA_MODEL = "amalia:9b-q4_k_m"
#
# Se ficar como None, o notebook tenta detectar
# automaticamente um modelo cujo nome contenha "amalia".
AMALIA_MODEL = "hf.co/ruialexrib/AMALIA-9B-0626-SFT-GGUF:Q3_K_M"

TEMPERATURE = 0.1

print(f"Ollama:     {OLLAMA_URL}")
print(f"Qdrant:     {QDRANT_URL}")
print(f"Embeddings: {EMBEDDING_MODEL}")
print(f"Collection: {COLLECTION_NAME}")
print(f"Top-K:      {TOP_K}")


Ollama:     http://localhost:11434
Qdrant:     http://localhost:6333
Embeddings: bge-m3
Collection: clara_bge_m3
Top-K:      5


## 1. Verificar Ollama e detectar AMALIA


## 2. RAG Configuration

The main runtime parameters of the end-to-end RAG pipeline are defined here.

These include the Ollama endpoint, Qdrant endpoint, embedding model, AMALIA model, vector collection, Top-K retrieval value, generation temperature, and request timeouts.

The configuration should remain aligned with the settings used by the production CLARA API.

In [3]:
def get_installed_models() -> list[str]:
    response = requests.get(
        f"{OLLAMA_URL}/api/tags",
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    return [
        model["name"]
        for model in data.get("models", [])
    ]


installed_models = get_installed_models()

print("Modelos instalados no Ollama:\n")

for model in installed_models:
    print(f"- {model}")


Modelos instalados no Ollama:

- bge-m3:latest
- hf.co/ruialexrib/AMALIA-9B-0626-SFT-GGUF:Q3_K_M
- gemma3:4b
- qwen3:4b
- mistral:latest
- qwen2.5:3b
- llama3.1:latest
- llama3:latest


## 2. RAG Configuration

The main runtime parameters of the end-to-end RAG pipeline are defined here.

These include the Ollama endpoint, Qdrant endpoint, embedding model, AMALIA model, vector collection, Top-K retrieval value, generation temperature, and request timeouts.

The configuration should remain aligned with the settings used by the production CLARA API.

In [4]:
def detect_amalia_model(
    installed_models: list[str]
) -> str | None:

    candidates = [
        model
        for model in installed_models
        if "amalia" in model.lower()
    ]

    if not candidates:
        return None

    # Se existirem vários, escolhe o primeiro.
    # Podes sempre substituir manualmente AMALIA_MODEL.
    return candidates[0]


if AMALIA_MODEL is None:
    AMALIA_MODEL = detect_amalia_model(
        installed_models
    )

if AMALIA_MODEL is None:
    raise RuntimeError(
        "Não foi encontrado nenhum modelo AMALIA no Ollama. "
        "Instala/importa o modelo ou define manualmente AMALIA_MODEL."
    )

print(f"AMALIA seleccionado: {AMALIA_MODEL}")


AMALIA seleccionado: hf.co/ruialexrib/AMALIA-9B-0626-SFT-GGUF:Q3_K_M


## 2. Verificar Qdrant


## 4. Qdrant Collection Validation

The notebook verifies that the Qdrant service and the CLARA vector collection are available.

The collection contains the legal chunks and semantic vectors generated during the ingestion pipeline and is the source of grounding evidence for answer generation.

In [5]:
def check_qdrant() -> bool:
    try:
        response = requests.get(
            f"{QDRANT_URL}/collections/{COLLECTION_NAME}",
            timeout=10
        )

        response.raise_for_status()
        return True

    except Exception as e:
        print(f"Erro ao contactar o Qdrant: {e}")
        return False


qdrant_ok = check_qdrant()

print(
    "Qdrant OK."
    if qdrant_ok
    else "Qdrant indisponível."
)


Qdrant OK.


## 3. Gerar embedding da pergunta


## 2. RAG Configuration

The main runtime parameters of the end-to-end RAG pipeline are defined here.

These include the Ollama endpoint, Qdrant endpoint, embedding model, AMALIA model, vector collection, Top-K retrieval value, generation temperature, and request timeouts.

The configuration should remain aligned with the settings used by the production CLARA API.

In [6]:
def generate_embedding(
    text: str,
    model: str = EMBEDDING_MODEL
) -> list[float]:

    if not text or not text.strip():
        raise ValueError(
            "O texto não pode estar vazio."
        )

    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={
            "model": model,
            "input": text
        },
        timeout=120
    )

    response.raise_for_status()

    data = response.json()

    embeddings = data.get("embeddings")

    if not embeddings:
        raise ValueError(
            "O Ollama não devolveu embeddings."
        )

    return embeddings[0]


## 4. Recuperação vetorial no Qdrant


## 5. Question Embedding

This function converts the user's natural-language question into a semantic vector using **BGE-M3**.

The same embedding model was used to represent the legal chunks during ingestion, allowing the question vector to be compared directly with the vectors stored in Qdrant.

In [7]:
def vector_search(
    question: str,
    top_k: int = 5,
    document_id: str | None = None
) -> list[dict]:

    query_vector = generate_embedding(
        question
    )

    payload = {
        "vector": query_vector,
        "limit": top_k,
        "with_payload": True,
        "with_vector": False
    }

    if document_id is not None:
        payload["filter"] = {
            "must": [
                {
                    "key": "document_id",
                    "match": {
                        "value": document_id
                    }
                }
            ]
        }

    response = requests.post(
        (
            f"{QDRANT_URL}/collections/"
            f"{COLLECTION_NAME}/points/search"
        ),
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    return response.json()["result"]


## 5. Construção do contexto

O LLM recebe apenas os chunks recuperados. Cada bloco inclui diploma, artigo, epígrafe e páginas.


## 7. Context Construction

The retrieved legal chunks are transformed into the context supplied to AMALIA.

The context combines the relevant legal text with source metadata so that the language model receives both the substantive provisions and information identifying their origin.

This is the central grounding mechanism of the RAG architecture.

In [8]:
def build_context(
    results: list[dict]
) -> str:

    blocks = []

    for rank, result in enumerate(
        results,
        start=1
    ):

        payload = result["payload"]

        header = (
            f"[Fonte {rank}]\n"
            f"Documento: {payload['document_id']}\n"
            f"Artigo: {payload['article']}\n"
        )

        if payload.get("article_title"):
            header += (
                f"Epígrafe: "
                f"{payload['article_title']}\n"
            )

        header += (
            f"Páginas: "
            f"{payload['page_start']}"
            f"-{payload['page_end']}\n"
        )

        block = (
            f"{header}\n"
            f"{payload['text']}"
        )

        blocks.append(block)

    return (
        "\n\n"
        + ("\n\n" + "-" * 80 + "\n\n").join(blocks)
    ).strip()


## 6. Prompt de sistema

O prompt força o AMALIA a:
- responder apenas com base no contexto recuperado;
- não preencher lacunas com conhecimento externo;
- indicar quando não existe informação suficiente;
- responder em português de Portugal;
- citar diploma e artigo sempre que possível.


## 8. Generation Instructions

The system prompt defines the behavioural constraints applied to AMALIA when generating a legal response.

The model is instructed to rely exclusively on the retrieved context, avoid inventing missing legal information, identify relevant legal sources whenever possible, answer in European Portuguese, and explicitly acknowledge when the retrieved evidence is insufficient.

These instructions reduce the risk of unsupported generation, although they cannot eliminate it completely.

In [9]:
SYSTEM_PROMPT = """És o CLARA, um assistente especializado na consulta de legislação, códigos e regulamentos portugueses.

Responde exclusivamente com base no contexto fornecido.

Regras:
1. Não utilizes conhecimento externo para completar informação em falta.
2. Não inventes artigos, números, taxas, datas, conceitos ou exceções.
3. Se o contexto não contiver informação suficiente para responder, indica claramente: "Não foi encontrada informação suficiente no contexto recuperado para responder com segurança."
4. Responde em português de Portugal.
5. Identifica, sempre que possível, o diploma e o artigo relevante.
6. Quando existirem várias fontes relevantes, distingue-as claramente.
7. Mantém a resposta objetiva e juridicamente prudente.
8. Não apresentes o conteúdo como aconselhamento jurídico profissional.
"""


## 7. Chamada ao AMALIA via Ollama


## 2. RAG Configuration

The main runtime parameters of the end-to-end RAG pipeline are defined here.

These include the Ollama endpoint, Qdrant endpoint, embedding model, AMALIA model, vector collection, Top-K retrieval value, generation temperature, and request timeouts.

The configuration should remain aligned with the settings used by the production CLARA API.

In [10]:
def generate_answer(
    question: str,
    context: str,
    model: str = AMALIA_MODEL,
    temperature: float = TEMPERATURE
) -> str:

    user_prompt = f"""Pergunta:
{question}

Contexto recuperado:
{context}

Responde à pergunta utilizando exclusivamente o contexto acima.
"""

    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": model,
            "stream": False,
            "messages": [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            "options": {
                "temperature": temperature
            }
        },
        timeout=300
    )

    response.raise_for_status()

    data = response.json()

    return data["message"]["content"].strip()


## 8. Função RAG completa


## 7. Context Construction

The retrieved legal chunks are transformed into the context supplied to AMALIA.

The context combines the relevant legal text with source metadata so that the language model receives both the substantive provisions and information identifying their origin.

This is the central grounding mechanism of the RAG architecture.

In [11]:
def ask_clara(
    question: str,
    top_k: int = TOP_K,
    document_id: str | None = None
) -> dict:

    start_time = time.perf_counter()

    results = vector_search(
        question=question,
        top_k=top_k,
        document_id=document_id
    )

    retrieval_time = (
        time.perf_counter()
        -
        start_time
    )

    context = build_context(
        results
    )

    generation_start = time.perf_counter()

    answer = generate_answer(
        question=question,
        context=context
    )

    generation_time = (
        time.perf_counter()
        -
        generation_start
    )

    sources = []

    for rank, result in enumerate(
        results,
        start=1
    ):

        payload = result["payload"]

        sources.append({
            "rank": rank,
            "score": result["score"],
            "document_id": payload[
                "document_id"
            ],
            "article": payload[
                "article"
            ],
            "article_title": payload.get(
                "article_title"
            ),
            "page_start": payload[
                "page_start"
            ],
            "page_end": payload[
                "page_end"
            ],
            "chunk_id": payload[
                "chunk_id"
            ]
        })

    return {
        "question": question,
        "answer": answer,
        "context": context,
        "sources": sources,
        "retrieval_seconds": round(
            retrieval_time,
            3
        ),
        "generation_seconds": round(
            generation_time,
            3
        ),
        "total_seconds": round(
            retrieval_time
            +
            generation_time,
            3
        )
    }


## 9. Teste completo


## 11. End-to-End RAG Test

A natural-language legal question is submitted to the complete CLARA pipeline.

This test validates the interaction between all runtime components: BGE-M3, Qdrant, context construction, AMALIA, and source metadata.

In [12]:
question = (
    "Como são tributados os rendimentos prediais?"
)

response = ask_clara(
    question=question,
    top_k=5
)

print("PERGUNTA")
print("-" * 80)
print(response["question"])

print("\nRESPOSTA")
print("-" * 80)
print(response["answer"])

print("\nTEMPOS")
print("-" * 80)
print(
    f"Retrieval: "
    f"{response['retrieval_seconds']} s"
)
print(
    f"Geração:   "
    f"{response['generation_seconds']} s"
)
print(
    f"Total:     "
    f"{response['total_seconds']} s"
)


PERGUNTA
--------------------------------------------------------------------------------
Como são tributados os rendimentos prediais?

RESPOSTA
--------------------------------------------------------------------------------
Os rendimentos prediais são tributados no âmbito da categoria F do IRS (Categoria F), conforme definido no Artigo 8.º do CIRS. Esta tributação aplica-se a:

1. **Rendimentos de arrendamento**: Importâncias relativas à cedência do uso do prédio ou de parte dele, aluguer de maquinismos e mobiliários, diferença entre a renda recebida do subarrendatário e paga ao senhorio, cedência do uso de bens imóveis para fins especiais (ex.: publicidade), cedência de partes comuns em regime de propriedade horizontal e constituição/cessação de direitos reais de gozo temporários sobre prédios rústicos ou urbanos.

2. **Rendimentos de arrendamento para habitação duradoura**: Importâncias relativas a contratos de direito real de habitação duradoura, tributadas desde o seu recebimento

## 10. Fontes recuperadas


### RAG Processing Step 13

This block performs an additional operation in the end-to-end CLARA workflow.

It contributes to connecting semantic retrieval, legal context construction, local LLM generation, source traceability, or runtime validation.

In [13]:
df_sources = pd.DataFrame(
    response["sources"]
)

df_sources[
    [
        "rank",
        "score",
        "document_id",
        "article",
        "article_title",
        "page_start",
        "page_end",
        "chunk_id"
    ]
]


,rank,score,document_id,article,article_title,page_start,page_end,chunk_id
0,1,0.704147,CIRS,Artigo 8.º,Rendimentos da categoria F,32,33,CIRS_ART_0027_CHUNK_001
1,2,0.656335,EBF,Artigo 45.º-C,"(Aditado pelo Decreto-Lei n.º 97/2026, de 20 d...",79,81,EBF_ART_0128_CHUNK_001
2,3,0.651471,CIRS,Artigo 16.º,As modificações do Código,11,23,CIRS_ART_0017_CHUNK_013
3,4,0.638217,EBF,Artigo 74.º-A,Transferência de imóveis de alojamento local p...,106,107,EBF_ART_0183_CHUNK_001
4,5,0.631915,CIRS,Artigo 72.º,Taxas especiais,86,91,CIRS_ART_0108_CHUNK_002


## 11. Inspecionar o contexto enviado ao AMALIA


### RAG Processing Step 14

This block performs an additional operation in the end-to-end CLARA workflow.

It contributes to connecting semantic retrieval, legal context construction, local LLM generation, source traceability, or runtime validation.

In [14]:
print(
    response["context"][:8000]
)


[Fonte 1]
Documento: CIRS
Artigo: Artigo 8.º
Epígrafe: Rendimentos da categoria F
Páginas: 32-33

1 - Consideram-se rendimentos prediais as rendas dos prédios rústicos, urbanos e mistos pagas ou colocadas à disposição dos respetivos titulares, quando estes não optarem pela sua tributação no âmbito da categoria B. 2 - São havidas como rendas: 32 a) As importâncias relativas à cedência do uso do prédio ou de parte dele e aos serviços relacionados com aquela cedência; b) As importâncias relativas ao aluguer de maquinismos e mobiliários instalados no imóvel locado; c) A diferença, auferida pelo sublocador, entre a renda recebida do subarrendatário e a paga ao senhorio; d) As importâncias relativas à cedência do uso, total ou parcial, de bens imóveis, para quaisquer fins especiais, designadamente publicidade; e) As importâncias relativas à cedência do uso de partes comuns de prédios em regime de propriedade horizontal; f) As importâncias relativas à constituição, a título oneroso, de direit

## 12. Pesquisa restringida a um diploma

Exemplo: responder utilizando apenas o CIRS.


## 11. End-to-End RAG Test

A natural-language legal question is submitted to the complete CLARA pipeline.

This test validates the interaction between all runtime components: BGE-M3, Qdrant, context construction, AMALIA, and source metadata.

In [15]:
response_cirs = ask_clara(
    question=(
        "Como são tributados "
        "os rendimentos prediais?"
    ),
    top_k=5,
    document_id="CIRS"
)

print(response_cirs["answer"])

pd.DataFrame(
    response_cirs["sources"]
)[
    [
        "rank",
        "score",
        "document_id",
        "article",
        "article_title"
    ]
]


Os rendimentos prediais são tributados no âmbito da categoria F (rendimentos prediais) do IRS, conforme definido no Artigo 8.º do CIRS. Estes incluem:

1. **Rendas de prédios rústicos, urbanos e mistos**: As importâncias relativas à cedência do uso do prédio ou de parte dele, bem como os serviços relacionados com essa cedência (alínea a).
2. **Aluguer de maquinismos e mobiliários** instalados no imóvel locado (alínea b).
3. **Diferença auferida pelo sublocador** entre a renda recebida do subarrendatário e a paga ao senhorio (alínea c).
4. **Cessão de uso, total ou parcial, de bens imóveis** para fins especiais, designadamente publicidade (alínea d).
5. **Cessão de partes comuns de prédios em regime de propriedade horizontal** (alínea e).
6. **Constituição, a título oneroso, de direitos reais de gozo temporários** sobre prédios rústicos, urbanos ou mistos (alínea f).
7. **Indemnizações que visem compensar perdas de rendimentos desta categoria** (alínea g).
8. **Rendimentos decorrentes d

,rank,score,document_id,article,article_title
0,1,0.704147,CIRS,Artigo 8.º,Rendimentos da categoria F
1,2,0.651471,CIRS,Artigo 16.º,As modificações do Código
2,3,0.631915,CIRS,Artigo 72.º,Taxas especiais
3,4,0.629810,CIRS,Artigo 72.º,Taxas especiais
4,5,0.598820,CIRS,Artigo 41.º,Deduções


## 13. Teste de ausência de informação

É importante testar se o modelo recusa responder quando o contexto não suporta a pergunta.


## 11. End-to-End RAG Test

A natural-language legal question is submitted to the complete CLARA pipeline.

This test validates the interaction between all runtime components: BGE-M3, Qdrant, context construction, AMALIA, and source metadata.

In [16]:
unsupported_question = (
    "Qual é a população de Lisboa em 2026?"
)

unsupported_response = ask_clara(
    question=unsupported_question,
    top_k=5
)

print(
    unsupported_response["answer"]
)


Não foi encontrada informação suficiente no contexto recuperado para responder com segurança.


## 14. Função simples para futura interface de chat

A interface web poderá chamar apenas esta função.


## 10. Complete CLARA RAG Pipeline

This function orchestrates the complete question-answering workflow.

For each question, CLARA:

1. generates the question embedding;
2. retrieves the Top-K legal chunks from Qdrant;
3. constructs the legal context;
4. sends the grounded prompt to AMALIA;
5. returns the generated answer;
6. preserves the retrieved sources;
7. records retrieval and generation times.

This orchestration logic forms the conceptual basis of the runtime implementation in `src/rag.py`.

In [17]:
def chat(
    question: str,
    document_id: str | None = None
) -> dict:
    return ask_clara(
        question=question,
        top_k=TOP_K,
        document_id=document_id
    )


### 11.1 Generated Answer

The answer produced by AMALIA is displayed for qualitative inspection.

The response should be assessed not only for linguistic quality but, more importantly, for consistency with the legal evidence retrieved from Qdrant.

In [18]:
# Exemplo
result = chat(
    "O que é o valor patrimonial tributário?"
)

print(result["answer"])


O valor patrimonial tributário é o valor que serve de base para a tributação do Imposto Municipal sobre Imóveis (IMI). É determinado por diferentes métodos consoante o tipo e a classificação do imóvel, conforme estabelecido no Código do IMI (CIMI):

- Para **prédios rústicos**, corresponde ao produto do seu rendimento fundiário pelo fator 20, arredondado para a dezena de euros imediatamente superior.  
- Para **prédios urbanos**, o valor patrimonial tributário pode ser determinado por:  
  - aplicação das regras previstas no CIMI (artigos 7.º, 135.º-C e outros);  
  - avaliação da parte principal do prédio, tendo em conta a valorização resultante das partes acessórias; ou  
  - soma dos valores das suas partes rústica e urbana, conforme as regras específicas para prédios mistos (artigo 7.º).  
- Para **prédios da espécie «Outros»**, o valor patrimonial tributário é determinado nos termos do artigo 46.º, podendo ser utilizado o método do custo adicionado do valor do terreno ou o valor u

## Resultado

O pipeline RAG está agora completo:

`Pergunta → embedding → Qdrant → contexto → AMALIA → resposta`

O passo seguinte será mover esta lógica dos notebooks para módulos Python, por exemplo:

```text
src/
├── embeddings.py
├── retrieval.py
├── llm.py
└── rag.py
```

Depois, a interface de chat poderá chamar apenas `ask_clara()` ou `chat()`.


## From Notebook Prototype to CLARA Application

This notebook completes the experimental RAG pipeline. The same main responsibilities are separated into reusable Python modules in the CLARA application:

```text
src/
├── config.py       # Runtime configuration
├── embeddings.py   # Query embedding generation
├── retrieval.py    # Qdrant semantic retrieval
├── llm.py          # AMALIA interaction and generation
└── rag.py          # End-to-end RAG orchestration
```

The application layer exposes this pipeline through **FastAPI**:

```text
app/main.py
```

The API provides both a native CLARA endpoint and an OpenAI-compatible interface, allowing **Open WebUI** to use CLARA as a conversational model.

The deployed architecture therefore becomes:

```text
Open WebUI
     │
     ▼
FastAPI / OpenAI-Compatible API
     │
     ▼
CLARA RAG Pipeline
     │
     ├────────► BGE-M3 / Ollama
     │
     ├────────► Qdrant
     │
     └────────► AMALIA-9B / Ollama
```

## Final Remarks

The eight notebooks collectively provide a reproducible path from raw Portuguese legal PDF documents to a locally deployed Retrieval-Augmented Generation system.

The complete pipeline covers:

```text
PDF Extraction
      ↓
Legal Parsing
      ↓
Article-Aware Chunking
      ↓
Embedding Generation
      ↓
Qdrant Indexing
      ↓
Semantic Search
      ↓
Retrieval Evaluation
      ↓
RAG Generation with AMALIA
```

The notebooks are primarily intended for corpus construction, experimentation, validation, and reproducibility. Interactive use of CLARA is handled by the modular Python application and its Docker-based deployment.
